# SMS ham preparation for Dataset 1

This notebook prepares **SMS ham** (`data/raw/human_legit/sms_ham.jsonl`) for the v2 Core human-side gathering stage.

Per `v2/docs/dataset_design_final.md`:
- `scenario_family = legitimate_sms`
- `fraudness = legitimate`, `channel = sms`, `label = 0` (human)

Output:
- `v2/data/interim/gathered/sms_ham_prepared.jsonl`

Notes:
- This source is treated as **legacy** (`time_band = legacy`).

In [1]:
from __future__ import annotations

import json
import re
import sys
import unicodedata
from pathlib import Path

import pandas as pd
from langdetect import detect, LangDetectException, DetectorFactory

DetectorFactory.seed = 0  # reproducibility

REPO = Path('/Users/askar/projects/antifraud-deepfake-detection')
BASE = REPO / 'v2'

sys.path.insert(0, str(BASE))
from config import length_bin as compute_length_bin

RAW_PATH = REPO / 'data/raw/human_legit/sms_ham.jsonl'
OUT_DIR = BASE / 'data/interim/gathered'
OUT_PATH = OUT_DIR / 'sms_ham_prepared.jsonl'

URL_RE = re.compile(r'(?i)(?:https?://\S+|www\.\S+|\b[a-z0-9][a-z0-9-]*\.[a-z]{2,}(?:/\S*)?)')
TOKEN_RE = re.compile(r'\w+|[^\w\s]', re.UNICODE)


def normalize_text(text: str) -> str:
    text = '' if text is None else str(text)
    text = unicodedata.normalize('NFKC', text)
    text = text.replace('\uFFFD', ' ')
    text = re.sub(r'[\x00-\x1F\x7F]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def mask_url(text: str) -> str:
    text = URL_RE.sub('[URL]', text)
    text = re.sub(r'https?://\S*', '[URL]', text)
    return text


def is_english(text: str) -> bool:
    if not text or len(text.strip()) < 10:
        return False
    try:
        return detect(text[:2000]) == 'en'
    except LangDetectException:
        return False


def token_length(text: str) -> int:
    return len(TOKEN_RE.findall(text))


print('RAW:', RAW_PATH)
print('OUT:', OUT_PATH)

RAW: /Users/askar/projects/antifraud-deepfake-detection/data/raw/human_legit/sms_ham.jsonl
OUT: /Users/askar/projects/antifraud-deepfake-detection/v2/data/interim/gathered/sms_ham_prepared.jsonl


In [2]:
# Load, normalize, mask URLs, language filter, deduplicate
rows = [json.loads(l) for l in RAW_PATH.read_text(encoding='utf-8').splitlines() if l.strip()]
print('raw rows:', len(rows))

texts_raw = [r.get('text', '') for r in rows]

df = pd.DataFrame({'text_raw': texts_raw})

df['text_norm'] = df['text_raw'].map(normalize_text)
df = df[df['text_norm'] != ''].copy()

df['text'] = df['text_norm'].map(mask_url).map(normalize_text)
df = df[df['text'] != ''].copy()

# Basic gates
df = df[df['text'].str.len() >= 5].copy()
df['token_length'] = df['text'].map(token_length)
df = df[df['token_length'] <= 5000].copy()

# Language filter
df['is_english'] = df['text'].map(is_english)
df_en = df[df['is_english']].copy()

grp = df_en.groupby('text', sort=False)
prepared = grp.first().reset_index(drop=False)
prepared['n_duplicate_rows'] = grp.size().values

print('after cleaning:', len(df))
print('after english filter:', len(df_en))
print('after dedup:', len(prepared))

raw rows: 4827
after cleaning: 4809
after english filter: 4297
after dedup: 4061


In [3]:
# Assign Core schema fields and write output
prepared['char_length'] = prepared['text'].str.len()
prepared['length_bin'] = prepared['token_length'].map(lambda t: compute_length_bin(int(t), 'sms'))
prepared['time_band'] = 'legacy'

prepared['label'] = 0
prepared['label_str'] = 'human'
prepared['fraudness'] = 'legitimate'
prepared['channel'] = 'sms'
prepared['scenario_family'] = 'legitimate_sms'
prepared['source_family'] = 'sms_ham'
prepared['dataset_source'] = 'sms_ham.jsonl'
prepared['origin_model'] = 'human'
prepared['split'] = 'tbd'
prepared['lang_filter_method'] = 'langdetect_v1'

out_cols = [
    'text', 'label', 'label_str', 'fraudness', 'channel', 'scenario_family',
    'source_family', 'dataset_source', 'time_band', 'length_bin',
    'char_length', 'token_length',
    'origin_model', 'split', 'lang_filter_method', 'n_duplicate_rows',
]

out_df = prepared[out_cols].copy()
OUT_DIR.mkdir(parents=True, exist_ok=True)
out_df.to_json(OUT_PATH, orient='records', lines=True, force_ascii=False)

print('written rows:', len(out_df))
print('output:', OUT_PATH)
print('\nlength_bin distribution:')
print(out_df['length_bin'].value_counts())

written rows: 4061
output: /Users/askar/projects/antifraud-deepfake-detection/v2/data/interim/gathered/sms_ham_prepared.jsonl

length_bin distribution:
length_bin
short     2555
medium    1441
long        65
Name: count, dtype: int64


In [4]:
# Validation
check = pd.read_json(OUT_PATH, lines=True)

REQUIRED_FIELDS = {
    'text', 'label', 'label_str', 'fraudness', 'channel',
    'scenario_family', 'source_family', 'dataset_source',
    'time_band', 'length_bin', 'origin_model', 'split',
}
missing = REQUIRED_FIELDS - set(check.columns)
assert not missing, f'Missing required fields: {missing}'

assert check['text'].astype(str).str.strip().ne('').all()
assert (check['label'] == 0).all()
assert (check['label_str'] == 'human').all()
assert (check['fraudness'] == 'legitimate').all()
assert (check['channel'] == 'sms').all()
assert (check['scenario_family'] == 'legitimate_sms').all()
assert (check['time_band'] == 'legacy').all()
assert set(check['length_bin']).issubset({'short', 'medium', 'long'})
assert check['text'].str.contains(r'https?://', regex=True).sum() == 0

print('Validation passed.')
print('Rows:', len(check))

Validation passed.
Rows: 4061
